In [1]:
import os, json, time

In [2]:
from openai import OpenAI
import utils

key_file = 'openai-api-key.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = OpenAI(  
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [3]:
model_name = 'o3-mini'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'o3-mini-2025-01-31'

In [4]:
data_dir = '../data/final_dataset'
long_ans_files = ['certamen_translation_long.json', 
                  'junior_scholarship_translation_long.json', 
                  'prosody_caesura_scansion_english.json',
                  'prosody_caesura_scansion_latin.json',
                  'prosody_feet_questions_english.json',
                  'prosody_feet_questions_latin.json'
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_translation_long.json 916
junior_scholarship_translation_long.json 350
prosody_caesura_scansion_english.json 41
prosody_caesura_scansion_latin.json 41
prosody_feet_questions_english.json 20
prosody_feet_questions_latin.json 20


In [5]:
def construct_long_ans_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    question_text += '\n' + utils.long_ans_format_instructions

    return question_text

In [6]:
prompt = construct_long_ans_user_prompt(file_to_data['certamen_translation_long.json'][0])
prompt

'Translate the motto of Alabama: Audēmus iūra nostra dēfendere.\nAt the end of your response, give your answer as:\nAnswer: answer text'

In [8]:
response = client.responses.create(
  model=model_endpoint,
  instructions = utils.sys_prompt,
  input = prompt,
  #temperature=0.6, 
  #top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.output_text)

"Audēmus iūra nostra dēfendere" translates as "We dare to defend our rights." 

Answer: We dare to defend our rights.


In [11]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [12]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    for q_dict in data:
        q_id = q_dict['question_id']
        prompt = construct_long_ans_user_prompt(q_dict)
        
        try:
            response = client.responses.create(
                model=model_endpoint,
                instructions = utils.sys_prompt,
                input = prompt,
            )
            resp = response.output_text
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.01)
        

        if i % 100 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)

certamen_translation_long.json
  0 / 930
  100 / 930
  200 / 930
  300 / 930
  400 / 930
  500 / 930
  600 / 930
  700 / 930
  800 / 930
  900 / 930
junior_scholarship_translation_long.json
  0 / 350
  100 / 350
  200 / 350
  300 / 350
prosody_caesura_scansion_english.json
  0 / 41
prosody_caesura_scansion_latin.json
  0 / 41
prosody_feet_questions_english.json
  0 / 20
prosody_feet_questions_latin.json
  0 / 20


redo feet questions

In [6]:
with_instr = True
data_dir = '../data/final_dataset'
long_ans_files = [
                  'prosody_feet_questions_english.json',
                  'prosody_feet_questions_latin.json'
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

prosody_feet_questions_english.json 20
prosody_feet_questions_latin.json 20


In [7]:
def construct_feet_user_prompt(q_dict):
    q_lang = q_dict['question_language']
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']

    if with_instr and q_lang == 'english':
        question_text += '\n' + utils.feet_eng_instr
    elif with_instr and q_lang == 'latin':
        question_text += '\n' + utils.feet_lat_instr

    question_text += '\n' + utils.long_ans_format_instructions

    return question_text

In [8]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [10]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0

    if with_instr:
        filename = filename.replace('.json', '_with_instr.json')

    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
        
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_feet_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.1)
        

        if i % 10 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)

prosody_feet_questions_english.json
  0 / 20
  10 / 20
prosody_feet_questions_latin.json
  0 / 20
  10 / 20


In [2]:
with open ('../data/final_dataset/prosody_caesura_scansion_english.json', 'r') as f:
    data = json.load(f)

In [3]:
import pandas as pd

df = pd.DataFrame(data)
df.to_csv('caesura_eng.tsv', sep='\t', index=None)